# LSTM Classifier for Article Bias Prediction

This notebook builds a deep learning model using LSTM (Long Short-Term Memory) networks to classify news articles by their political bias (left, center, right).

**Data Source**: Article-Bias-Prediction dataset with JSON articles
**Model**: LSTM with embedding layer
**Task**: Multi-class text classification

In [ ]:
import os
import json
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder

import pickle

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Load and Parse JSON Data

Load all JSON files from the Article-Bias-Prediction dataset and convert to a structured DataFrame.

In [ ]:
# Set paths relative to the Bias Classifier project root.
# Run this notebook from C:/jh/python2/Bias Classifier or another checkout root.
PROJECT_ROOT = Path.cwd()
data_dir = PROJECT_ROOT / 'Article-Bias-Prediction-main' / 'data' / 'jsons'
if not data_dir.exists():
    raise FileNotFoundError(
        f"Training JSON directory not found: {data_dir}. "
        "Restore Article-Bias-Prediction-main/data/jsons from the local dataset before running."
    )
json_files = glob.glob(str(data_dir / '*.json'))
print(f"Found {len(json_files)} JSON files")

# Load all JSON files into a list
data_list = []
for json_file in json_files:
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            article = json.load(f)
            data_list.append(article)
    except Exception as e:
        print(f"Error loading {json_file}: {e}")

print(f"Successfully loaded {len(data_list)} articles")

# Convert to DataFrame
df = pd.DataFrame(data_list)
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst row:\n{df.iloc[0]}")

## 2. Exploratory Data Analysis

Analyze the structure, class distribution, and text statistics of the dataset.

In [ ]:
# Check bias_text distribution
print("Bias Distribution:")
print(df['bias_text'].value_counts())
print(f"\nBias percentages:\n{df['bias_text'].value_counts(normalize=True) * 100}")

# Visualize bias distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['bias_text'].value_counts().plot(kind='bar', ax=axes[0], color=['red', 'blue', 'green'])
axes[0].set_title('Bias Distribution')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Bias')

df['bias_text'].value_counts(normalize=True).plot(kind='bar', ax=axes[1], color=['red', 'blue', 'green'])
axes[1].set_title('Bias Distribution (Normalized)')
axes[1].set_ylabel('Proportion')
axes[1].set_xlabel('Bias')
plt.tight_layout()
plt.show()

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Text statistics
df['content_length'] = df['content'].fillna('').apply(len)
df['word_count'] = df['content'].fillna('').apply(lambda x: len(x.split()))

print("\nText statistics:")
print(f"Average content length: {df['content_length'].mean():.0f} characters")
print(f"Average word count: {df['word_count'].mean():.0f} words")
print(f"Min/Max content length: {df['content_length'].min()}/{df['content_length'].max()}")
print(f"Min/Max word count: {df['word_count'].min()}/{df['word_count'].max()}")

# Show word count distribution by bias
fig, ax = plt.subplots(figsize=(10, 5))
for bias in df['bias_text'].unique():
    data = df[df['bias_text'] == bias]['word_count']
    ax.hist(data, alpha=0.5, label=bias, bins=30)
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.set_title('Word Count Distribution by Bias')
ax.legend()
plt.show()

# Sample articles
print("\nSample articles by bias:")
for bias in df['bias_text'].unique():
    sample = df[df['bias_text'] == bias]['content'].iloc[0][:200]
    print(f"\n{bias}: {sample}...")

## 3. Preprocess Text Data

Clean and normalize the text data for model input.

In [ ]:
import re
import string

def preprocess_text(text):
    """Clean and preprocess text"""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
print("Preprocessing text data...")
df['text_cleaned'] = df['content'].apply(preprocess_text)

# Remove empty texts
df = df[df['text_cleaned'].str.len() > 0].reset_index(drop=True)
print(f"Articles after removing empty texts: {len(df)}")

# Encode labels
label_encoder = LabelEncoder()
df['bias_encoded'] = label_encoder.fit_transform(df['bias_text'])

print("\nLabel mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {i}: {label}")

print(f"\nSample preprocessed text:\n{df['text_cleaned'].iloc[0][:300]}...")
print(f"\nLabel: {df['bias_text'].iloc[0]} -> {df['bias_encoded'].iloc[0]}")

## 4. Tokenize and Encode Text

Convert text to sequences of integers and pad to fixed length.

In [ ]:
# Parameters
MAX_WORDS = 5000  # Vocabulary size
MAX_SEQUENCE_LENGTH = 500  # Max length of sequence

# Create tokenizer
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['text_cleaned'].values)

# Convert text to sequences
X = tokenizer.texts_to_sequences(df['text_cleaned'].values)

# Pad sequences
X = pad_sequences(X, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

# Prepare labels
y = df['bias_encoded'].values

print(f"Tokenizer vocabulary size: {len(tokenizer.word_index)}")
print(f"Sequence shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nSequence length statistics:")
print(f"  Max: {len(tokenizer.word_index)}")
print(f"  Vocab size used: {min(MAX_WORDS, len(tokenizer.word_index))}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Class distribution in train set: {np.bincount(y_train)}")
print(f"Class distribution in test set: {np.bincount(y_test)}")

## 5. Build LSTM Model Architecture

Design the neural network with embedding, bidirectional LSTM, and dense layers.

In [ ]:
# Model parameters
EMBEDDING_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.5
NUM_CLASSES = len(label_encoder.classes_)

# Build model
model = Sequential([
    # Embedding layer
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH,
        name='embedding'
    ),
    
    # Dropout for regularization
    Dropout(DROPOUT_RATE),
    
    # Bidirectional LSTM
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True), name='bidirectional_lstm_1'),
    Dropout(DROPOUT_RATE),
    
    # Second LSTM layer
    Bidirectional(LSTM(LSTM_UNITS), name='bidirectional_lstm_2'),
    Dropout(DROPOUT_RATE),
    
    # Dense layers
    Dense(128, activation='relu', name='dense_1'),
    Dropout(DROPOUT_RATE),
    
    Dense(64, activation='relu', name='dense_2'),
    Dropout(DROPOUT_RATE),
    
    # Output layer
    Dense(NUM_CLASSES, activation='softmax', name='output')
])

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## 6. Train the LSTM Classifier

Compile and train the model on the training data.

In [ ]:
# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Train model
print("Training model...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining completed!")

## 7. Evaluate Model Performance

Assess the model using various metrics and visualizations.

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Make predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f}")

# Classification report
print("\nClassification Report:")
class_names = label_encoder.classes_
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 8. Make Predictions on New Data

Use the trained model to predict bias on new articles.

In [ ]:
def predict_bias(text, model, tokenizer, label_encoder, max_length=500):
    """Predict bias of a new article"""
    # Preprocess
    cleaned_text = preprocess_text(text)
    
    # Tokenize
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')
    
    # Predict
    prediction = model.predict(padded, verbose=0)
    predicted_class = np.argmax(prediction[0])
    confidence = prediction[0][predicted_class]
    predicted_label = label_encoder.inverse_transform([predicted_class])[0]
    
    return {
        'text': text[:200],
        'predicted_bias': predicted_label,
        'confidence': confidence,
        'probabilities': {label: float(pred) for label, pred in zip(label_encoder.classes_, prediction[0])}
    }

# Test on some articles from the dataset
print("Predictions on test set articles:\n")
for i in range(5):
    idx = np.random.randint(0, len(X_test))
    article_text = df.iloc[idx]['content']
    actual_bias = df.iloc[idx]['bias_text']
    
    result = predict_bias(article_text, model, tokenizer, label_encoder)
    
    print(f"Article {i+1}:")
    print(f"  Text: {result['text']}...")
    print(f"  Actual bias: {actual_bias}")
    print(f"  Predicted bias: {result['predicted_bias']}")
    print(f"  Confidence: {result['confidence']:.4f}")
    print(f"  Probabilities: {result['probabilities']}")
    print()

# Test on custom article
custom_article = """
The government's new environmental policy shows promising progress in addressing climate change. 
Scientists agree that these measures are crucial for our future. The administration has demonstrated 
a strong commitment to sustainability and renewable energy sources.
"""

print("\nPrediction on custom article:")
result = predict_bias(custom_article, model, tokenizer, label_encoder)
print(f"Text: {result['text']}...")
print(f"Predicted bias: {result['predicted_bias']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"Probabilities: {result['probabilities']}")

## 9. Save Model and Artifacts

Save the trained model and preprocessing artifacts for future use.

In [ ]:
# Create save directory under the Bias Classifier project root
save_dir = PROJECT_ROOT / 'bias_lstm_model'
save_dir.mkdir(parents=True, exist_ok=True)

# Save model
model_path = save_dir / 'bias_lstm_model.keras'
model.save(str(model_path))
print(f"Model saved to {model_path}")

# Save tokenizer
tokenizer_path = save_dir / 'tokenizer.pkl'
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to {tokenizer_path}")

# Save label encoder
encoder_path = save_dir / 'label_encoder.pkl'
with open(encoder_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder saved to {encoder_path}")

# Save metadata
metadata = {
    'max_words': MAX_WORDS,
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'num_classes': NUM_CLASSES,
    'class_names': list(label_encoder.classes_),
    'accuracy': float(accuracy),
    'model_parameters': {
        'embedding_dim': EMBEDDING_DIM,
        'lstm_units': LSTM_UNITS,
        'dropout_rate': DROPOUT_RATE
    }
}

metadata_path = save_dir / 'metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to {metadata_path}")

print(f"\nAll artifacts saved to {save_dir}")